# 🏥 Healthcare Supply Chain Optimization

This notebook demonstrates how to use machine learning and data analysis techniques to optimize inventory, reduce waste, and ensure timely replenishment of medical supplies.

## 📦 Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')


## 🗂️ Step 2: Load and Explore Inventory Dataset (Synthetic)

In [ ]:
# Sample synthetic inventory data
data = {
    "Item": ["Masks", "Gloves", "Sanitizer", "Ventilators", "Gowns"] * 12,
    "Month": list(range(1, 13)) * 5,
    "Inventory": np.random.randint(500, 2000, 60),
    "Demand": np.random.randint(400, 1800, 60),
    "Replenishment_LeadTime": np.random.randint(3, 14, 60)
}
df = pd.DataFrame(data)
df.head()


## 📉 Step 3: Visualize Inventory Trends

In [ ]:
plt.figure(figsize=(10, 6))
for item in df['Item'].unique():
    item_df = df[df['Item'] == item]
    plt.plot(item_df['Month'], item_df['Inventory'], label=item)
plt.xlabel("Month")
plt.ylabel("Inventory Level")
plt.title("Monthly Inventory Trends")
plt.legend()
plt.show()


## 🤖 Step 4: Predict Future Demand

In [ ]:
df['Lagged_Demand'] = df['Demand'].shift(1).fillna(method='bfill')
X = df[['Lagged_Demand', 'Replenishment_LeadTime']]
y = df['Demand']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")


## 📊 Step 5: Feature Importance

In [ ]:
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
feature_importance.plot(kind='barh', title='Feature Importance')
plt.show()


## ✅ Summary

We demonstrated a basic supply chain optimization workflow using synthetic data, including demand forecasting and feature analysis. In practice, you can integrate this with real-time inventory APIs or ERP systems.

## Gradio Dashboard for Supply Chain Forecasting

In [ ]:

import gradio as gr
import pandas as pd
import joblib

# Load the trained model and sample input features
model = joblib.load("supply_model.pkl")

def forecast(demand, delivery_time, criticality_score):
    input_df = pd.DataFrame([{
        'average_daily_demand': demand,
        'delivery_lead_time': delivery_time,
        'criticality_score': criticality_score
    }])
    prediction = model.predict(input_df)[0]
    return f"Suggested stock reorder point: {prediction:.2f}"

gr.Interface(
    fn=forecast,
    inputs=[
        gr.Number(label="Average Daily Demand"),
        gr.Number(label="Delivery Lead Time (days)"),
        gr.Slider(1, 10, step=1, label="Criticality Score")
    ],
    outputs=gr.Textbox(label="Forecasted Reorder Point"),
    title="Supply Chain Optimization Assistant",
    description="Forecasts reorder points based on key inventory metrics."
).launch()
